In [10]:
import re
import emoji
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from slang_dict import abbreviations

In [11]:

# Download necessary NLTK resources
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger")


class Preprocessing:
    """
    Text Preprocessing class for cleaning text data.
    Handles:
    - Removing special characters
    - Handling emojis
    - Expanding abbreviations
    - Removing URLs, HTML tags
    - Lemmatization
    - Stopword removal
    """

    def clean_string(self, text):
        """Removes special characters and extra whitespace."""
        text = re.sub(r"[^A-Za-z0-9\s]", "", text)  # Remove special characters
        text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces
        return text

    def convert_abbrev(self, text):
        """Expands abbreviations using predefined dictionary."""
        return " ".join(
            [abbreviations.get(word.lower(), word) for word in text.split()]
        )

    def remove_urls(self, text):
        """Removes URLs from the text."""
        return re.sub(r"https?://\S+|www\.\S+", "", text)

    def remove_html(self, text):
        """Removes HTML tags from the text."""
        return re.sub(r"<.*?>", "", text)

    def lemmatize_text(self, text):
        """Lemmatizes text using NLTK's WordNetLemmatizer."""
        lemmatizer = WordNetLemmatizer()
        words = word_tokenize(text)
        return " ".join([lemmatizer.lemmatize(word) for word in words])

    def remove_stopwords(self, text):
        """Removes common English stopwords."""
        stop_words = set(stopwords.words("english"))
        words = word_tokenize(text)
        return " ".join([word for word in words if word.lower() not in stop_words])

    def handle_emoji(self, text):
        """Converts emojis to their textual representation."""
        return emoji.demojize(text)


def preprocess_column(df, column_name, preprocessor):
    """
    Applies a sequence of preprocessing steps to a specified column in a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column_name (str): Column to preprocess.
        preprocessor (Preprocessing): Instance of Preprocessing class.

    Returns:
        pd.Series: Cleaned text column.
    """
    return (
        df[column_name]
        .astype(str)
        .apply(preprocessor.clean_string)
        .apply(preprocessor.remove_urls)
        .apply(preprocessor.remove_html)
        .apply(preprocessor.handle_emoji)
        .apply(preprocessor.lemmatize_text)
        .apply(preprocessor.remove_stopwords)
    )

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [12]:


# Load CSV file
csv_file_path = "Siemens_Error_codes_cleaned.csv"
df = pd.read_csv(csv_file_path)

In [13]:
print(df.columns)

Index(['category', 'error_code', 'description', 'remedy', 'error_type'], dtype='object')


In [14]:

# Initialize Preprocessor
preprocessor = Preprocessing()

# Apply preprocessing to both 'description' and 'remedy' columns
df["description_cleaned"] = preprocess_column(df, "description", preprocessor)
df["remedy_cleaned"] = preprocess_column(df, "remedy", preprocessor)

In [15]:

# Save the cleaned dataset
df.to_csv("error_codes_cleaned.csv", index=False)
print("Preprocessing complete. Cleaned data saved to 'error_codes_cleaned.csv'.")

Preprocessing complete. Cleaned data saved to 'error_codes_cleaned.csv'.
